# Fundamentals of Machine Learning — Exercise 9
## Support vector machines and hyperparameter tuning with Optuna

**Format:** independent work, **90 minutes**. We take one model, the **support vector machine (SVM)**, and learn how to find its good settings **automatically and honestly** with **Optuna**. You will see that a model with default settings can be far worse than a plain linear regression, and that a tuned one is only as trustworthy as your evaluation.

### Three kinds of tasks
| | What to do |
|---|---|
| 🔮 **Predict** | Write your guess **before** you run the next cell. We keep a scoreboard at the end. |
| 🛠️ **Do** | A short `TODO` (1–6 lines) followed by a ✅/❌ self-check. |
| 🕵️ **Think** | A puzzle or a decision. Answer in your own words and **compare with your neighbour**. |

### The story of today's exercise
We have one question and answer it step by step: **is a support vector machine better than a plain linear regression on our data, and can we trust the answer?**

1. **Baseline and a locked test set** (Part 1): we measure the simple model first and hide the test data.
2. **The default SVM is worse** than the baseline (Part 2). Why? Its settings (`C`, `gamma`) do not fit our data.
3. **Finding better settings by hand is slow** (Part 3), and a grid explodes. We need a smarter search: **Optuna**.
4. **Optuna in action** (Parts 4–5): it tunes the SVM, we read the result carefully and discover that our search range was too narrow. We also check that Optuna really beats random search.
5. **Can we trust the number?** (Part 6): the best score of many trials is optimistic, so we check it honestly on the locked test set.
6. **Why does the tuned SVM only match the linear regression?** (Part 7): we let Optuna choose the kernel and see that the data are almost linear.

### How to work
1. Complete the `TODO`s; variables set to `None` are waiting for you. Run the **check** cell below.
2. Stuck for more than 5 minutes? Open the 💡 hint. Still stuck? Continue and write the problem down at the end.
3. Run the notebook **top to bottom** (cells depend on each other). Some cells run several seconds: that is a model being tuned, not a freeze.

## 0. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVR, SVC

plt.rcParams["axes.grid"] = True
RANDOM_STATE = 42

# Optuna is not part of scikit-learn. If it is missing, try to install it automatically.
try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"])
    import optuna        # if the installation failed, you get an error here: run  %pip install optuna  and restart the kernel
from optuna.importance import get_param_importances
optuna.logging.set_verbosity(optuna.logging.WARNING)      # do not print a line for every trial


def check(test, ok="Correct.", hint="Not yet."):
    """Self-check. `test` is a function returning True/False (an error counts as a failure)."""
    try:
        passed = bool(test())
    except Exception as e:
        passed = False
        hint = hint + f"  [{type(e).__name__}: {e}]"
    print(("✅ " + ok) if passed else ("❌ " + hint))

## 1. Parameters, hyperparameters, data and a baseline 

### 1.1 Two kinds of numbers in a model
Consider an SVM. Some numbers are **learned from the training data** (*model parameters*, for example the support vectors that define the fitted curve). Others are **chosen by you** before or around training and control how the learning behaves (*hyperparameters*, for example `C`). Optuna tunes the second kind.

**Activity 1.** Write *parameter*, *hyperparameter* or *evaluation choice* for each item:

| Item | Type |
|---|---|
| `C=10` | ? |
| the fitted support vectors | ? |
| the RBF kernel | ? |
| the weights of a fitted linear regression | ? |
| 5-fold cross-validation | ? |
| R² as the metric | ? |

✍️ **Your answer:** *(double-click this cell and write here)*

### 1.2 The data, and a locked test set
Today we use one data set: **diabetes** – 442 patients, 10 measurements (age, BMI, blood pressure, blood serum values, …), target = a number describing the disease progression after one year. It is a **regression** problem: a table of features → a number.

We measure quality with the **cross-validated R²** (5 folds): 1 = perfect, **0 = as good as always predicting the mean**, negative = worse than the mean.

**First thing: split off a test set and lock it.** From now on we use only the development part `X_dev`, `y_dev` for everything (training, tuning, comparing). The test part `X_test`, `y_test` is touched **once**, in Part 6. Looking at it earlier would make our final number dishonest (Part 6 shows why).

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
cv = KFold(5, shuffle=True, random_state=RANDOM_STATE)        # the same 5 folds for every comparison today
print("development:", X_dev.shape, "| locked test set:", X_test.shape, "| target: mean %.0f, std %.0f" % (y.mean(), y.std()))

**Always start with a baseline.** A baseline is a very simple model that any more complex model must beat; if it does not, the complexity is not worth it. Here the baseline is the plain **linear regression** (a straight-line-like model: the prediction is a weighted sum of the features).

In [ ]:
baseline_lr = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()), X_dev, y_dev, cv=cv, scoring="r2").mean()
print(f"baseline – linear regression: CV R² = {baseline_lr:.3f}")

## 2. SVM in a nutshell 
A **support vector machine** is the topic of the lecture; here is all we need today. We use its **regression** version, `SVR`: it fits a smooth function through the data. The behaviour is set by three choices:

| Hyperparameter | What it does | Too small | Too large |
|---|---|---|---|
| `C` | how strongly errors on the training points are punished | a very smooth, simple function (underfit) | the curve follows every point (overfit) |
| `kernel` | the *shape* of the function: `linear` (a straight line / plane), `rbf` (a smooth flexible curve), `poly` (a polynomial) | – | – |
| `gamma` (only for `rbf` and `poly`) | how **local** the curve is: how far the influence of one point reaches | an almost straight, very smooth curve | a wiggly curve, each point acts alone (overfit) |

(The regression SVM has one more setting, `epsilon`, the width of a "tube" around the curve inside which errors cost nothing. We keep its default, `0.1`.)

Practical point: SVM works with **distances**, so the features must be **scaled**. We always use the pipeline `StandardScaler → SVR`, as in Exercise 8 (the pipeline fits the scaler only on the training part of each fold).

🔮 **Predict.** The SVM with the **default settings** (`C=1`, `kernel="rbf"`, `gamma` chosen automatically), compared with the plain linear regression baseline: will it be **better**, about the **same** (± 0.05) or **worse**?

In [ ]:
pred_default = None      # "better", "same" or "worse"

In [ ]:
default_svr = cross_val_score(make_pipeline(StandardScaler(), SVR()), X_dev, y_dev, cv=cv, scoring="r2").mean()
actual_default = "better" if default_svr > baseline_lr + 0.05 else ("worse" if default_svr < baseline_lr - 0.05 else "same")
print(f"linear regression: {baseline_lr:.3f}   |   SVR with the default settings: {default_svr:.3f}")
print(f"🔮 your guess: {pred_default!r}   actual: {actual_default!r}   {'✓' if pred_default == actual_default else '✗'}")

**Question 1 🕵️.** The default SVM is much worse than a linear regression. Using the table above: which hyperparameter is the main suspect, and in which direction would you move it?

✍️ **Your answer:** *(double-click this cell and write here)*

## 3. Why is tuning hard? 

How do we find better values? By hand, by a **grid** (all combinations of a few values), by **random search**, or with **Optuna** (it **learns from the previous trials** where good values are):

| Method | Idea | Problem |
|---|---|---|
| by hand | try, look, try again | slow, depends on your mood, hard to repeat |
| **grid search** | try **all combinations** of a few values per hyperparameter | the number of combinations **explodes** |
| **random search** | try random combinations | wasteful: it does not learn from earlier tries |
| **Optuna** (Bayesian optimization, TPE) | learns from the previous trials and tries promising values next | needs a budget of trials; the result is only as honest as your evaluation |

First, "by hand". Below are five settings of `C` and `gamma`. For each we look at the score on the **training** data (the model has seen it) and the **validation** score (cross-validation, unseen data). Run it and look.

In [ ]:
manual = [(0.1, 0.001), (1, 0.1), (10, 0.01), (100, 0.01), (1000, 0.1)]
rows = []
for C, g in manual:
    r = cross_validate(make_pipeline(StandardScaler(), SVR(C=C, gamma=g)), X_dev, y_dev, cv=cv, scoring="r2", return_train_score=True)
    rows.append({"setting": f"C={C:g}, gamma={g:g}", "train R²": r["train_score"].mean(), "validation R²": r["test_score"].mean()})
manual_df = pd.DataFrame(rows).set_index("setting")
manual_df.plot(kind="bar", figsize=(8, 3.6)); plt.axhline(baseline_lr, color="gray", ls="--"); plt.ylim(-0.1, 1.05)
plt.xticks(rotation=20, ha="right"); plt.title("by hand: training vs validation score (dashed: linear regression)"); plt.tight_layout(); plt.show()

**Question 2 🕵️.** (a) Which setting underfits (weak on both), and which overfits (strong on training, weak on validation)? (b) Does the largest `C` give the best validation score? (c) Why is "change one number at a time" not enough to find the best setting?

✍️ **Your answer:** *(double-click this cell and write here)*

🔮 **Predict.** A grid with **20 values for each of `C` and `gamma`**, every combination evaluated by 5-fold cross-validation: how many **seconds** would it take on your computer? Write your guess, then measure how long **one** cross-validated evaluation takes.

In [ ]:
pred_grid_seconds = None     # <- your guess in seconds

In [ ]:
t0 = time.time()
cross_val_score(make_pipeline(StandardScaler(), SVR()), X_dev, y_dev, cv=cv)
t_one = time.time() - t0
print(f"one cross-validated evaluation takes {t_one:.3f} s")

### 🛠️ Task 3.1
How many combinations does the grid have (20 values for each of 2 hyperparameters, `n_grid2`)? How many would it have with **5** hyperparameters (`n_grid5`)? How many **hours** would the 5-hyperparameter grid take if one evaluation takes `t_one` seconds (`hours5`)?

<details><summary>💡 Hint</summary>

`n_grid2 = 20 ** 2`; `n_grid5 = 20 ** 5`; `hours5 = n_grid5 * t_one / 3600`

</details>

In [ ]:
n_grid2 = None
n_grid5 = None
hours5 = None

In [ ]:
check(lambda: n_grid2 == 400 and n_grid5 == 3_200_000, "400 and 3,200,000 combinations.", "20 values for each hyperparameter: 20 ** number_of_hyperparameters.")
check(lambda: abs(hours5 - n_grid5 * t_one / 3600) < 1e-9, f"2 hyperparameters: about {n_grid2 * t_one:.0f} s (your guess: {pred_grid_seconds}). 5 hyperparameters: about {hours5:.0f} hours.", "hours5 = n_grid5 * t_one / 3600")
print("A fast model and a tiny data set. A slower model or more hyperparameters, and a grid becomes impossible. Today we use 40 trials: about %.0f seconds." % (40 * t_one))

## 4. Optuna in a nutshell 

| Word | Meaning |
|---|---|
| **objective** | a function that takes a `trial`, builds and evaluates a model and **returns one number** (e.g. the cross-validated R²) |
| **trial** | one try = one set of hyperparameter values. Inside the objective you ask the trial for values: `trial.suggest_int(...)`, `trial.suggest_float(...)`, `trial.suggest_categorical(...)` |
| **study** | the whole optimization: `optuna.create_study(direction="maximize")` (or `"minimize"`) and `study.optimize(objective, n_trials=30)` |
| **sampler** | the algorithm that chooses the next values. The default `TPESampler` learns from the previous trials (its **first 10 trials are random** exploration) |
| result | `study.best_params`, `study.best_value`, `study.trials_dataframe()` |

The simplest example: find the `x` that minimizes `(x − 2)²`.

In [ ]:
def objective_toy(trial):
    x = trial.suggest_float("x", -10, 10)
    return (x - 2) ** 2

study_toy = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_toy.optimize(objective_toy, n_trials=20)
print("best x:", round(study_toy.best_params["x"], 3), "| best value:", round(study_toy.best_value, 4))
display(study_toy.trials_dataframe()[["number", "params_x", "value"]].head(6).round(3))

Notice: **the objective is just a function** returning a number, and Optuna decides which `x` to try next. In a real problem the number is a cross-validated score and the trial values are hyperparameters.

In Part 5 you write such an objective yourself, for the SVM.

## 5. Tune the SVM with Optuna 

Now `C` and `gamma` **together**, with the `rbf` kernel. The search space:

| Hyperparameter | Range | `suggest_` call |
|---|---|---|
| `C` | 0.1 … 10 | `suggest_float("C", 0.1, 10, log=True)` |
| `gamma` | 0.0001 … 1 | `suggest_float("gamma", 1e-4, 1, log=True)` |

`log=True` because these values change by **orders of magnitude**: 0.0001, 0.001, 0.01, … should be tried equally often. (The defaults, `C=1` and `gamma=0.1`, lie inside the ranges.)

🔮 **Predict.** By how much can the best of **40 Optuna trials** beat the default SVR (in R² points)?

In [ ]:
pred_gain = None     # "less than 0.05", "0.05 to 0.2" or "more than 0.2"

### 🛠️ Task 5.1 – the objective
Write `objective(trial, Xd=X_dev, yd=y_dev, c_range=(0.1, 10))`: suggest `C` (within `c_range`, log scale) and `gamma` (0.0001 … 1, log scale), build `make_pipeline(StandardScaler(), SVR(**params))` and **return the mean cross-validated R²**. (The extra arguments let us reuse the function later with another range.)

<details><summary>💡 Hint</summary>

`params = dict(C=trial.suggest_float(\"C\", *c_range, log=True), gamma=trial.suggest_float(\"gamma\", 1e-4, 1, log=True))`; `model = make_pipeline(StandardScaler(), SVR(**params))`; `return cross_val_score(model, Xd, yd, cv=cv, scoring=\"r2\").mean()`

</details>

In [ ]:
def objective(trial, Xd=X_dev, yd=y_dev, c_range=(0.1, 10)):
    # TODO: suggest C (use *c_range) and gamma
    params = dict()
    model = make_pipeline(StandardScaler(), SVR(**params))
    # TODO: return the mean cross-validated R²
    return None

In [ ]:
class _FakeTrial:                     # a quick test of your function without Optuna: it takes the middle of every range
    def suggest_int(self, n, lo, hi): return (lo + hi) // 2
    def suggest_float(self, n, lo, hi, log=False): return (lo * hi) ** 0.5 if log else (lo + hi) / 2
check(lambda: -1 < objective(_FakeTrial()) < 1, "The objective returns a sensible number (an R²).", "The objective must return one number: the mean cross-validated R².")

### 🛠️ Task 5.2 – the study
Create the study (`direction="maximize"`, `TPESampler(seed=RANDOM_STATE)`) and run **40 trials** (the first 10 are random exploration, the other 30 are guided by what Optuna has learned). `study.enqueue_trial(defaults)` makes **trial 0 = the default settings**, so we always know the baseline.

<details><summary>💡 Hint</summary>

`study = optuna.create_study(direction=\"maximize\", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))`, then `study.enqueue_trial(defaults)` and `study.optimize(objective, n_trials=40)` (a few seconds).

</details>

In [ ]:
defaults = dict(C=1.0, gamma=0.1)
study = None            # TODO: create the study
# study.enqueue_trial(defaults)       <- uncomment after you create the study
# TODO: run 40 trials

In [ ]:
check(lambda: len(study.trials) == 40, "40 trials were run.", "study.optimize(objective, n_trials=40)")
default_value = study.trials[0].value
gain = study.best_value - default_value
check(lambda: study.best_value >= default_value, f"Default (trial 0): R² = {default_value:.3f}; best of 40: R² = {study.best_value:.3f}  (+{gain:.3f}).", "Did you enqueue the defaults as the first trial?")
actual_gain = "less than 0.05" if gain < 0.05 else ("0.05 to 0.2" if gain <= 0.2 else "more than 0.2")
print(f"🔮 your guess: {pred_gain!r}   actual: {actual_gain!r}   {'✓' if pred_gain == actual_gain else '✗'}")
print(f"for comparison: linear regression baseline R² = {baseline_lr:.3f}")
print("\nbest hyperparameters:", {k: round(v, 4) for k, v in study.best_params.items()})

df = study.trials_dataframe()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(df["number"], df["value"], "o", color="lightgray", label="trial")
ax[0].plot(df["number"], df["value"].cummax(), "-", color="C0", label="best so far")
ax[0].axhline(baseline_lr, color="gray", ls="--", label="linear regression")
ax[0].set_ylim(max(df["value"].min(), -0.3), None)
ax[0].set_xlabel("trial"); ax[0].set_ylabel("CV R²"); ax[0].set_title("optimization history"); ax[0].legend(fontsize=8)
sc = ax[1].scatter(df["params_C"], df["params_gamma"], c=df["value"].clip(lower=0), cmap="viridis")
ax[1].set_xscale("log"); ax[1].set_yscale("log"); ax[1].set_xlabel("C (log scale)"); ax[1].set_ylabel("gamma (log scale)")
ax[1].set_title("where the trials went (colour = CV R²)"); plt.colorbar(sc, ax=ax[1])
plt.tight_layout(); plt.show()

**Question 3 🕵️.** (a) Compare the three numbers: the default SVM, the tuned SVM and the linear regression baseline. What do you conclude? (b) Look at the **right** picture. Where do the best (bright) trials sit, and what is the largest `C` that was allowed? Does anything look **suspicious**? How would you test your suspicion? (c) Why `log=True`? What would happen if `C` were sampled **uniformly** between 0.1 and 10? (d) Can you use the best value as the final estimate of how good the model is? (Part 6 checks it.)

✍️ **Your answer:** *(double-click this cell and write here)*

### 🛠️ Task 5.3 – is the search range too narrow?
A rule: if the best value of a hyperparameter lies **at the edge of its range**, the range was probably too narrow. Write `boundary_position(value, lo, hi)` returning where the value lies on the **log scale** of the range: **0** at `lo`, **1** at `hi`, 0.5 in the geometric middle (use `np.log`). Then compute `pos_C` for the best `C` of `study` (range 0.1 … 10).

<details><summary>💡 Hint</summary>

`(np.log(value) - np.log(lo)) / (np.log(hi) - np.log(lo))`; `pos_C = boundary_position(study.best_params[\"C\"], 0.1, 10)`

</details>

In [ ]:
def boundary_position(value, lo, hi):
    # TODO
    return None

pos_C = None

In [ ]:
check(lambda: abs(boundary_position(10, 0.1, 10) - 1) < 1e-9 and abs(boundary_position(1, 0.1, 10) - 0.5) < 1e-9, "boundary_position works: 0 = lower edge, 1 = upper edge.", "boundary_position(10, 0.1, 10) should be 1 and boundary_position(1, 0.1, 10) should be 0.5.")
print(f"best C lies at position {pos_C:.2f} of its range (0 = lower edge, 1 = upper edge)")
print("⚠️ at an edge: the range is probably too narrow" if (pos_C > 0.9 or pos_C < 0.1) else "inside the range: no warning from this rule")

### 🛠️ Task 5.4 – test your suspicion
Run a **second study** `study_wide` with the same objective but **`C` from 0.1 to 3000** (`c_range=(0.1, 3000)`), 40 trials, defaults enqueued first, a new TPE sampler with the same seed. Use `lambda t: objective(t, c_range=(0.1, 3000))`.

<details><summary>💡 Hint</summary>

`study_wide = optuna.create_study(direction=\"maximize\", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))`; `study_wide.enqueue_trial(defaults)`; `study_wide.optimize(lambda t: objective(t, c_range=(0.1, 3000)), n_trials=40)`

</details>

In [ ]:
study_wide = None
# TODO: create, enqueue the defaults, optimize

In [ ]:
check(lambda: len(study_wide.trials) == 40 and study_wide.best_value > study.best_value, "The wider range found a clearly better configuration.", "Run 40 trials with c_range=(0.1, 3000).")
pos_wide = boundary_position(study_wide.best_params["C"], 0.1, 3000)
print(pd.DataFrame({"range of C": ["0.1 – 10", "0.1 – 3000"],
                    "best CV R²": [study.best_value, study_wide.best_value],
                    "best C": [study.best_params["C"], study_wide.best_params["C"]],
                    "position in range": [pos_C, pos_wide]}).round(3).to_string(index=False))
print(f"linear regression baseline: {baseline_lr:.3f}")

**Question 4 🕵️.** (a) What changed when the range was widened? (b) Would you have noticed the problem if you had only looked at `best_params` and `best_value`? (c) Formulate a **rule** for yourself about search ranges.

✍️ **Your answer:** *(double-click this cell and write here)*

### 5.5 Near-ties, and which hyperparameter matters (run and read)
The ten best trials of the wide study, and Optuna's estimate of the **importance** of each hyperparameter:

In [ ]:
top = study_wide.trials_dataframe().sort_values("value", ascending=False).head(10)[["number", "value", "params_C", "params_gamma"]]
display(top.round(4))
imp = get_param_importances(study_wide)
plt.figure(figsize=(6, 2.0))
plt.barh(list(imp)[::-1], list(imp.values())[::-1]); plt.xlabel("importance"); plt.title("importance within the tested search space"); plt.show()

**Question 5 🕵️.** (a) The top trials differ in their score by very little but have **different** `C` and `gamma`. Can you claim that one exact pair is "the optimum"? (b) Which hyperparameter looks more important, and why must we add "within the tested search space"? Could a hyperparameter look unimportant just because its range was narrow? (c) Would another 100 trials necessarily change the conclusion?

✍️ **Your answer:** *(double-click this cell and write here)*

### 5.6 Is Optuna really better than random search?
Evaluating an SVM is fast, so we can do it properly: **5 repetitions** with different seeds for each sampler, 40 trials each (`TPESampler` against `RandomSampler`, which does not learn). Run and answer the next question.

In [ ]:
def best_curve(sampler_class, seed):
    st = optuna.create_study(direction="maximize", sampler=sampler_class(seed=seed))
    st.enqueue_trial(defaults)
    st.optimize(lambda t: objective(t, c_range=(0.1, 3000)), n_trials=40)
    return st.trials_dataframe()["value"].cummax().to_numpy()

curves = {"Optuna (TPE)": np.array([best_curve(optuna.samplers.TPESampler, s) for s in range(5)]),
          "random search": np.array([best_curve(optuna.samplers.RandomSampler, s) for s in range(5)])}
plt.figure(figsize=(7, 3.6))
for name, c in curves.items():
    plt.plot(c.mean(axis=0), label=name); plt.fill_between(range(40), c.min(axis=0), c.max(axis=0), alpha=0.2)
plt.ylim(0.3, None); plt.xlabel("trial"); plt.ylabel("best CV R² so far (mean and range of 5 runs)"); plt.legend(); plt.show()
print({name: round(c[:, -1].mean(), 3) for name, c in curves.items()})

**Question 6 🕵️.** Write a **cautious** conclusion: which method is better on average, by how much, with what budget and on how many repetitions? Why did we repeat it 5 times? What would you expect with 10 hyperparameters instead of 2?

✍️ **Your answer:** *(double-click this cell and write here)*

## 6. 🕵️ Honest evaluation: the winner's curse 

The best of the 40 trials was **chosen** because it scored best on these particular cross-validation folds. Among many attempts the best one is partly the luckiest one. It is the **same lesson as the leakage in Exercise 8**: the data we use to *choose* must not be the data we use to *judge*. How big can the effect be?

### 6.1 Tune on pure noise
Exercise 8, Part 4 again: **100 "patients", 20 random features, labels = coin flips**. No model can honestly be better than 50 %. We let Optuna tune an SVM classifier (`SVC`, the classification version of the SVM, with the same `C` and `gamma`) with 40 trials (cross-validated accuracy).

🔮 **Predict.** What will be the **best cross-validated accuracy** among the 40 trials?

In [ ]:
pred_noise = None      # "below 0.55", "0.55 to 0.65" or "above 0.65"

In [ ]:
rng = np.random.RandomState(1)
X_noise, y_noise = rng.randn(100, 20), rng.randint(0, 2, 100)
X_fresh, y_fresh = rng.randn(2000, 20), rng.randint(0, 2, 2000)      # 2000 NEW random patients (never used for tuning)
cv_noise = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

def objective_noise(trial):
    model = SVC(C=trial.suggest_float("C", 0.01, 1000, log=True), gamma=trial.suggest_float("gamma", 1e-4, 10, log=True))
    return cross_val_score(model, X_noise, y_noise, cv=cv_noise).mean()

study_noise = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_noise.optimize(objective_noise, n_trials=40)
best_cv_noise = study_noise.best_value
mean_cv_noise = study_noise.trials_dataframe()["value"].mean()

### 🛠️ Task 6.1
Take the best hyperparameters of this study (`study_noise.best_params`), fit `SVC(**best)` on `(X_noise, y_noise)` and store its **accuracy on the new patients** `(X_fresh, y_fresh)` in `test_acc_noise`.

<details><summary>💡 Hint</summary>

`model = SVC(**study_noise.best_params).fit(X_noise, y_noise)`; `test_acc_noise = model.score(X_fresh, y_fresh)`

</details>

In [ ]:
# TODO
test_acc_noise = None

In [ ]:
check(lambda: 0.4 < test_acc_noise < 0.6, f"Accuracy on new random patients: {test_acc_noise:.3f} – the guessing level, as it must be.", "Score the model on X_fresh, y_fresh (new patients).")
actual_noise = "below 0.55" if best_cv_noise < 0.55 else ("0.55 to 0.65" if best_cv_noise <= 0.65 else "above 0.65")
print(f"mean CV accuracy of all 40 trials: {mean_cv_noise:.3f}")
print(f"BEST cross-validated accuracy:     {best_cv_noise:.3f}   <- what the study 'found'")
print(f"accuracy on 2000 NEW patients:     {test_acc_noise:.3f}   <- the truth")
print(f"🔮 your guess: {pred_noise!r}   actual: {actual_noise!r}   {'✓' if pred_noise == actual_noise else '✗'}")

**Question 7 🕵️.** (a) The data contain nothing, yet the best trial reports a clearly better accuracy than 50 %. Why? (b) What does it have to do with Exercise 8, Part 4? (c) What would you tell a colleague who reports "after tuning, my model reaches X % in cross-validation"?

✍️ **Your answer:** *(double-click this cell and write here)*

### 6.2 The honest protocol on the diabetes data – the one look at the test set
**The protocol:**
1. **Split off a test set first** and do not touch it (we did that in Part 1).
2. Tune with cross-validation **on the development part only** (Part 5).
3. Choose the final configuration **using development data only**, refit on the development part and evaluate **once** on the test set, next to the defaults and the baseline.

### 🛠️ Task 6.2
Our final choice is the best configuration of `study_wide`. (1) Fit `make_pipeline(StandardScaler(), SVR(**study_wide.best_params))` on `(X_dev, y_dev)` and store its **test R²** in `r2_tuned`. (2) Do the same for the **default** `make_pipeline(StandardScaler(), SVR())` into `r2_default`, and for the linear regression `make_pipeline(StandardScaler(), LinearRegression())` into `r2_lr`.

<details><summary>💡 Hint</summary>

`r2_tuned = make_pipeline(StandardScaler(), SVR(**study_wide.best_params)).fit(X_dev, y_dev).score(X_test, y_test)`; the default and linear models are analogous.

</details>

In [ ]:
r2_tuned, r2_default, r2_lr = None, None, None

In [ ]:
check(lambda: all(v is not None for v in (r2_tuned, r2_default, r2_lr)) and r2_tuned > r2_default, "All three models were evaluated once on the test set.", "Fit on (X_dev, y_dev) and score on (X_test, y_test).")
print(pd.DataFrame({"R²": {
    "best CV value of the study (development data)": study_wide.best_value,
    "TEST: tuned SVM": r2_tuned,
    "TEST: default SVM": r2_default,
    "TEST: linear regression (baseline)": r2_lr}}).round(3).to_string())
print(f"test set size: {len(y_test)} patients")

**Question 8 🕵️.** (a) Compare the first line (the best CV value) with the test R² of the tuned SVM. Is the test value lower, as in 6.1? Why can it differ from the noise example? (b) Suppose the test R² of the tuned SVM had been **worse than the linear regression** and you were tempted to go back and widen the ranges or change the metric. Why would that be a mistake? (c) The test set has only 111 patients. How sure are you that the tuned SVM is better than the default one, and better than the linear regression?

✍️ **Your answer:** *(double-click this cell and write here)*

## 7. A search space with choices: kernel and preprocessing 

Optuna's strength: the search space does not have to be a table. You can let it **choose the kernel** and ask only for the hyperparameters that make sense for that choice (a *conditional* space), and even choose the **preprocessing**, for example whether to use **PCA** (Exercise 8) and with how many components.

The kernels: `linear` (no `gamma`), `rbf` (has `gamma`) and `poly` (has `gamma` and a `degree`). Everything is inside a pipeline, so nothing leaks.

### 🛠️ Task 7.1
Fill in the `poly` branch: it needs its own `gamma` (name `poly_gamma`, range 0.0001 … 0.1, log scale) and `degree` (integer 2 … 3). *(Ranges are limited on purpose: a polynomial kernel with a large `gamma` is very slow and unstable.)*

<details><summary>💡 Hint</summary>

`params[\"gamma\"] = trial.suggest_float(\"poly_gamma\", 1e-4, 0.1, log=True)`; `params[\"degree\"] = trial.suggest_int(\"poly_degree\", 2, 3)`. Use **different names** for the `gamma` of different kernels.

</details>

In [ ]:
def objective_multi(trial):
    steps = [StandardScaler()]
    if trial.suggest_categorical("use_pca", [True, False]):
        steps.append(PCA(n_components=trial.suggest_int("pca_components", 3, 10)))
    kernel = trial.suggest_categorical("kernel", ["rbf", "linear", "poly"])
    params = dict(kernel=kernel, C=trial.suggest_float("C", 0.1, 300, log=True))
    if kernel == "rbf":
        params["gamma"] = trial.suggest_float("rbf_gamma", 1e-4, 1, log=True)
    elif kernel == "poly":
        # TODO: gamma (poly_gamma) and degree (poly_degree)
        pass
    model = make_pipeline(*steps, SVR(**params))
    return cross_val_score(model, X_dev, y_dev, cv=cv, scoring="r2").mean()

In [ ]:
study_multi = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_multi.optimize(objective_multi, n_trials=60)
df_m = study_multi.trials_dataframe()
check(lambda: set(df_m["params_kernel"]) == {"rbf", "linear", "poly"} and df_m["params_poly_degree"].notna().any(), "All three kernels were tried, `poly` with its degree.", "Did the poly branch run? Check objective_multi.")
print(df_m.groupby("params_kernel")["value"].agg(["count", "max"]).round(3).rename(columns={"count": "trials", "max": "best CV R²"}).sort_values("best CV R²", ascending=False))
print(f"\nPCA used in the best trial: {study_multi.best_params['use_pca']}")
print("best:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in study_multi.best_params.items()})

**Question 9 🕵️.** (a) Which kernel won, and does it surprise you? Did the best trial use PCA? (b) Why would it be hard to do this with `GridSearchCV`? (c) Which part of this search space would you change if the data had 500 features and only 200 patients?

✍️ **Your answer:** *(double-click this cell and write here)*

## 8. Wrap-up 

### Optuna cheat sheet
```python
def objective(trial):
    value = trial.suggest_float("x", 0.001, 1, log=True)          # also suggest_int, suggest_categorical
    ...                                                            # build the model, evaluate with cross-validation
    return score

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.enqueue_trial({...defaults...})                              # start from the default settings
study.optimize(objective, n_trials=50)
study.best_params, study.best_value, study.trials_dataframe()
```

### The rules to remember
1. **Always have a baseline** (here: linear regression) and **start from the defaults**. For an SVM the defaults can be far from good, and a tuned model may only reach the baseline.
2. Use a sensible, not-too-wide search space; use `log=True` for hyperparameters that span orders of magnitude (`C`, `gamma`).
3. **Lock the test set first, tune with cross-validation on the development part, judge once on the test set.** The best cross-validated value of a study is optimistic (the winner's curse), and a decision made after seeing the test result spoils the test.
4. **Inspect the study, not only `best_params`:** is the best value at the edge of its range (widen it)? Are the top trials near-ties (do not claim one "optimum")? Is the importance only valid within the searched space?
5. Put preprocessing **inside the objective / pipeline** (Exercise 8), otherwise tuning leaks.
6. Fix the seeds, and repeat with several seeds when a difference is small. A few hundredths of R² on 111 patients is not a proof.

| | `GridSearchCV` | `RandomizedSearchCV` | Optuna |
|---|---|---|---|
| search space | fixed table | distributions | any Python code (conditional, per kernel or model) |
| learns from earlier trials | no | no | yes (TPE) |
| good for | 1–2 hyperparameters | a quick baseline | many hyperparameters, expensive models |

(Optuna can also stop bad trials early (*pruning*) and store the study in a database file (`storage="sqlite:///study.db"`) – we do not need it today.)

### 🔮 Scoreboard

In [ ]:
points = 0
points += (pred_default == actual_default)
points += (pred_gain == actual_gain)
points += (pred_noise == actual_noise)
print(f"Your prediction score: {points} / 3  (default SVM vs. baseline, gain of tuning, best CV accuracy on noise)")
print("0: surprised by everything - good, that is how you learn.  2–3: solid intuition for model behaviour.")

### Self-check
- [ ] I know what `C`, `kernel` and `gamma` do in an SVM, and why SVM needs scaled features.
- [ ] I can write an objective function, create a study and read `best_params`, `best_value` and the trial table.
- [ ] I know why the best cross-validated score of a study is optimistic and how to evaluate honestly.
- [ ] I know how to let Optuna choose a kernel (and a preprocessing step) together with its hyperparameters, without leakage.
- [ ] I always compare a tuned model with a simple baseline.
- [ ] I check whether the best value lies at the edge of its search range.
- [ ] I know why the test set is looked at once, after all decisions.